In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [5]:
# Sentences we want sentence embeddings for
sentences = ["people are running on tread machines in a gym with bright orange walls","Building muscle and strength","Practicing for a marathon"]

results = []

for sentence in sentences:
    # Load model from HuggingFace Hub
    tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
    model = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2')

    # Tokenize sentences
    encoded_input = tokenizer(sentence, padding=True, truncation=True, return_tensors='pt')

    # Compute token embeddings
    with torch.no_grad():
        model_output = model(**encoded_input)

    # Perform pooling
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

    # Normalize embeddings
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    results.append(sentence_embeddings)

print("Sentence embeddings:")
# print(sentence_embeddings)
print(len(results), "embeddings generated.")

Sentence embeddings:
3 embeddings generated.


In [7]:
a = results[0].numpy()
b = results[1].numpy()
c = results[2].numpy()

# a, b embedding space distance
distance_ab = torch.dist(torch.tensor(a), torch.tensor(b))
# a, c embedding space distance
distance_ac = torch.dist(torch.tensor(a), torch.tensor(c))

print("Distance between embedding a and b:", distance_ab)
print("Distance between embedding a and c:", distance_ac)

Distance between embedding a and b: tensor(1.3289)
Distance between embedding a and c: tensor(1.2147)


Distance between embedding a and b: tensor(1.0743)

Distance between embedding a and c: tensor(1.1487)